# Lesson 02 - Preskúmanie rámca Microsoft Agent

**Microsoft Agent Framework (MAF)** je jednotný rámec na vytváranie AI agentov. Poskytuje čistú, skladateľnú architektúru so štyrmi základnými stavebnými blokmi:

- **Klient** – pripája sa k endpointu AI modelu a spravuje komunikáciu
- **Agent** – obaluje klienta s inštrukciami a definíciami nástrojov
- **Nástroje** – rozširujú schopnosti agenta o vlastné funkcie, ktoré môže model volať
- **Relácia** – udržiava históriu konverzácie pre viackrokové interakcie

V tejto lekcii vytvoríme **agenta na rezerváciu cestovania**, ktorý kontroluje dostupnosť destinácií pomocou týchto konceptov.


## Nastavenie


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pochopenie architektúry rámca agenta

Microsoft Agent Framework nasleduje viacvrstvovú architektúru:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` sa pripája k Azure OpenAI nasadeniu. Spravuje overenie, formátovanie požiadaviek a analýzu odpovedí.
2. **Agent** – Vytvorený z klienta cez `provider.create_agent()`, agent kombinuje prístup k modelu s inštrukciami (systémový prompt) a nástrojmi.
3. **Nástroje** – Python funkcie dekorované `@tool`, ktoré môže agent vyvolať na vykonanie akcií alebo získanie dát.
4. **Sedenie** – Objekt `AgentSession` (vytvorený cez `agent.create_session()`), ktorý ukladá históriu konverzácie a umožňuje viackolovú komunikáciu, kde si agent pamätá predchádzajúci kontext.

Poďme si každý stupeň vybudovať krok za krokom.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pridávanie nástrojov s dekorátorom @tool

Nástroje umožňujú agentom vykonávať akcie nad rámec generovania textu. Dekorátor `@tool` konvertuje bežnú Python funkciu na niečo, čo môže agent zavolať.

Kľúčové body:
- Použite `Annotated[type, "description"]`, aby model rozumel každému parametru.
- Dokumentačný reťazec sa stane popisom nástroja, ktorý model vidí.
- `approval_mode="never_require"` znamená, že nástroj sa spustí automaticky bez potvrdenia používateľa.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytvorenie agenta s nástrojmi

Teraz skombinujeme klienta, inštrukcie a nástroje do agenta. `instructions` slúžia ako systémový prompt — definujú osobnosť a správanie agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Viackolové konverzácie s reláciami

`AgentSession` (vytvorené pomocou `agent.create_session()`) sleduje všetky správy v konverzácii. Tým, že odovzdáme tú istú reláciu do každého volania `agent.run()`, má agent prístup k celému históriu konverzácie a môže sa odvolať na predchádzajúce správy.

Odovzdávame `tools=[check_destination_availability]`, aby agent mohol pri každom kroku volať náš kontrolór dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Zhrnutie

V tejto lekcii ste preskúmali štyri piliere Microsoft Agent Frameworku:

| Koncept | Čo ste sa naučili |
|---------|--------------------|
| **Klient** | `AzureAIProjectAgentProvider` sa pripojí k Azure OpenAI pomocou autentifikácie na základe prihlasovacích údajov |
| **Agent** | `provider.create_agent()` zbalí pripojenie modelu s inštrukciami a menom |
| **Nástroje** | Dekorátor `@tool` sprístupňuje Python funkcie, ktoré môže agent volať |
| **Sedenie** | `agent.create_session()` uchováva históriu konverzácie počas viacerých kôl |

Tieto stavebné bloky spolu tvoria agentov, ktorí môžu viesť prirodzené rozhovory, volať externé funkcie a udržiavať kontext — základ pre pokročilejšie agentné vzory v neskorších lekciách.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o obmedzení zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, majte prosím na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo chybné interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
